# NewProcessing -- Experian / TRAIN

Re-runs the full NEW pipeline -- the denominator fix **plus the placeholder
change** -- over the SAME 400k experian train sample the modeling experiment
used (`samples/experian_train/app.parquet`), reading the train split's
`mapped/` chunks.

## The change

The experian asset declared `placeholder: "-"`, which deleted every dash from
the payment pattern before features were built. Per the Experian spec (CIS
Cross Reference Guide, Appendix T "Payment Profile Indicators", Segment
357.B4.5, p. 139) the dash is a REAL month -- "No update received" -- not
formatting. Stripping it shifted every older month one position more recent,
misaligning lookback windows and `months_since_*`, and shortening the string.
On the `payment_processor_change` branches the placeholder is removed: dashes
stay at their true calendar position and are excluded from the percent
denominators via `missing_data_chars` instead. This moves `percent_*`,
`number_*` (window membership shifts), `months_since_*`, and
`payment_history_length` (trailing dashes now subtract).

## Why we only re-run experian

- **experian** -- the placeholder removal is a real behavior change (above).
- **transunion** -- its `placeholder: "/"` removal is a no-op: the TU pattern
  character set (TU4.1 User Guide, Appendix C, pp. 838-840) contains no `/`
  and it never occurs in the data (verified; by-hand vs normal path was
  bit-identical). The TU `missing_data_chars` behavior was already in the
  experiment's NEW run.
- **equifax** -- nothing changed since the experiment's NEW run (placeholder
  kept, `missing_data_chars` already in).

Output (the NEW location `Build_Model_New_With_Change.ipynb` points at):

    payment_processing_research_data/new_normalized_and_processed/experian_train/normalized/
    payment_processing_research_data/new_normalized_and_processed/experian_train/processed/

Processed is saved WITH the ZEST_KEY index (no repeat of the lost-key bug).
Everything printed is also logged to `logs/new_processing_experian_train.log`.

In [1]:
# Install BOTH packages from the payment_processor_change branches.
# git+ssh because the Katlean repos are private and this box authenticates
# to GitHub over SSH (no HTTPS credentials). --no-deps so the rest of the
# env is untouched. RESTART THE KERNEL after this cell the first time you
# run it, so the new code is actually imported.
%pip install --quiet --no-deps --force-reinstall \
    "git+ssh://git@github.com/Katlean/feature-engine-parts.git@payment_processor_change" \
    "git+ssh://git@github.com/Katlean/model-engine.git@payment_processor_change"


Note: you may need to restart the kernel to use updated packages.


In [2]:
import gc
import inspect
from pathlib import Path

import pandas as pd

from feature_engine_parts.fe_parts_V2.preprocessors.payment_pattern_aggregator import PaymentPatternsAggregatorV2
from model_engine.assets.utils import load_asset
from model_engine.feature_engine_V2.listed_objects_engines import PreprocessorV2
from model_engine.feature_engine_V2.feature_engine import AggregationEngine

BUREAU     = 'experian'
ROLE       = 'train'
ASSET_PATH = 'experian/arf7/fe2/trade.json'
# fix_processing sampled the experian 'test' role from the VALID split.
ROLE_TO_SPLIT = {'train': 'train', 'test': 'valid'}
N_BUCKETS  = 50

DATA_DIR = Path('/home/jag/payment-processor-research/payment_processing_research_data')
OUT_ROOT = DATA_DIR / 'new_normalized_and_processed'

# --- verify the BRANCH code + asset are what's actually installed ---
sig = inspect.signature(PaymentPatternsAggregatorV2.__init__)
assert 'missing_data_chars' in sig.parameters, 'old feature-engine-parts still loaded -- restart the kernel'

asset = load_asset(ASSET_PATH)
ppa = next(s['params'] for s in asset['preprocess'] if s['type'] == 'PaymentPatternsAggregatorV2')
assert 'missing_data_chars' in ppa, 'old model-engine asset still loaded -- restart the kernel'
assert ppa['payment_patterns'].get('placeholder') is None, \
    'placeholder should be REMOVED for experian -- old model-engine still installed?'

agg_asset = load_asset('aggregation/fe2/trade.json')
print('missing_data_chars =', ppa['missing_data_chars'],
      '| placeholder =', ppa['payment_patterns'].get('placeholder'))


/home/jag/.conda/envs/model_engine_2_py310/lib/python3.10/site-packages/zaml/common/utils/io.py:17: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


missing_data_chars = ['-'] | placeholder = None


In [3]:
import os, sys, time
from datetime import datetime, timedelta

LOG_DIR = Path('/home/jag/payment-processor-research/logs')
LOG_DIR.mkdir(exist_ok=True)


class _Tee:
    """Mirror stdout to a log file so every print survives the session."""
    def __init__(self, *streams):
        self.streams = streams
    def write(self, data):
        for s in self.streams:
            try:
                s.write(data); s.flush()
            except Exception:
                pass
    def flush(self):
        for s in self.streams:
            try: s.flush()
            except Exception: pass


def _now():
    return datetime.now().strftime('%H:%M:%S')


def _fmt_secs(s):
    return str(timedelta(seconds=int(s)))


def clear_dir(d):
    d = Path(d)
    if d.exists():
        for f in d.glob('*.parquet'):
            f.unlink()
    d.mkdir(parents=True, exist_ok=True)


def _process_role(role):
    t0       = time.time()
    split    = ROLE_TO_SPLIT[role]
    keys     = set(pd.read_parquet(DATA_DIR / 'samples' / f'{BUREAU}_{role}' / 'app.parquet',
                                   columns=['ZEST_KEY'])['ZEST_KEY'])
    in_dir   = DATA_DIR / BUREAU / split / 'mapped'
    out_dir  = OUT_ROOT / f'{BUREAU}_{role}'
    norm_dir = out_dir / 'normalized'
    proc_dir = out_dir / 'processed'
    clear_dir(norm_dir)
    clear_dir(proc_dir)

    preprocessor = PreprocessorV2(api=asset['preprocess'])
    parts = sorted(in_dir.glob('part-*.parquet'))
    assert parts, f'no mapped chunks at {in_dir}'
    print(f'[{_now()}] ##### {BUREAU}/{role} #####')
    print(f'[{_now()}]   reading split={split!r} from {in_dir}')
    print(f'[{_now()}]   {len(keys):,} sample ZEST_KEYs | {len(parts)} mapped chunks')
    print(f'[{_now()}]   writing -> {norm_dir} and {proc_dir} (dirs cleared)')

    # PHASE 1 -- filter each mapped chunk to the sample keys, preprocess, save
    print(f'\n[{_now()}] phase 1: filter to sample keys + PreprocessorV2 '
          f'(new aggregator: missing_data_chars={ppa["missing_data_chars"]}, '
          f'placeholder={ppa["payment_patterns"].get("placeholder")!r})')
    t1 = time.time()
    found, rows_in, rows_kept, out_i = set(), 0, 0, 0
    for i, p in enumerate(parts):
        chunk      = pd.read_parquet(p)
        rows_in   += len(chunk)
        chunk      = chunk[chunk['ZEST_KEY'].isin(keys)]
        if len(chunk):
            normalized = preprocessor.transform(chunk)
            normalized.to_parquet(norm_dir / f'part-{out_i:05d}.parquet', index=False)
            found.update(chunk['ZEST_KEY'].unique())
            rows_kept += len(normalized)
            out_i     += 1
            del normalized
        del chunk
        gc.collect()
        if (i + 1) % 25 == 0 or i == len(parts) - 1:
            done    = i + 1
            elapsed = time.time() - t1
            rate    = done / elapsed
            eta     = (len(parts) - done) / rate if rate else 0
            print(f'[{_now()}]   chunk {done}/{len(parts)} '
                  f'| kept {rows_kept:,}/{rows_in:,} tradelines ({rows_kept / max(rows_in, 1):.1%}) '
                  f'| {len(found):,}/{len(keys):,} keys found '
                  f'| {rate:.1f} chunks/s, ETA {_fmt_secs(eta)}')

    coverage = len(found) / len(keys)
    print(f'[{_now()}] phase 1 done in {_fmt_secs(time.time() - t1)}: '
          f'{out_i} normalized parts, {rows_kept:,} tradelines, '
          f'key coverage {len(found):,}/{len(keys):,} ({coverage:.2%})')
    assert coverage > 0.99, f'only {coverage:.1%} of sample keys found -- wrong split for this role?'

    # PHASE 2 -- aggregate per ZEST_KEY hash bucket, KEEP the key as the index
    t2 = time.time()
    agg_eng = AggregationEngine(asset=agg_asset, table_name='trade')
    df = pd.read_parquet(norm_dir)
    print(f'\n[{_now()}] phase 2: AggregationEngine over {len(df):,} normalized rows, '
          f'{df["ZEST_KEY"].nunique():,} applicants, {N_BUCKETS} hash buckets')
    df['_bucket'] = (pd.util.hash_pandas_object(df['ZEST_KEY'], index=False) % N_BUCKETS).astype('int16')

    total, n_cols = 0, None
    for b in range(N_BUCKETS):
        sub = df[df['_bucket'] == b].drop(columns='_bucket')
        if not len(sub):
            continue
        processed = agg_eng.transform(sub)
        processed.to_parquet(proc_dir / f'part-{b:03d}.parquet', index=True)   # index=True: keep ZEST_KEY!
        total += len(processed)
        n_cols = processed.shape[1]
        if (b + 1) % 10 == 0 or b == N_BUCKETS - 1:
            elapsed = time.time() - t2
            eta     = elapsed / (b + 1) * (N_BUCKETS - b - 1)
            print(f'[{_now()}]   bucket {b + 1}/{N_BUCKETS} '
                  f'| {total:,} applicants aggregated | ETA {_fmt_secs(eta)}')
        del sub, processed
        gc.collect()
    del df
    gc.collect()
    print(f'[{_now()}] phase 2 done in {_fmt_secs(time.time() - t2)}: '
          f'{total:,} applicants x {n_cols:,} trade features -> {proc_dir}')
    print(f'[{_now()}] {BUREAU}/{role} TOTAL elapsed: {_fmt_secs(time.time() - t0)}')


def process_role(role):
    """Run _process_role with everything printed ALSO written to
    logs/new_processing_<bureau>_<role>.log (line-buffered, survives a
    kernel death mid-run)."""
    log_path = LOG_DIR / f'new_processing_{BUREAU}_{role}.log'
    original_stdout = sys.stdout
    with open(log_path, 'w', buffering=1) as log_file:
        sys.stdout = _Tee(original_stdout, log_file)
        try:
            print(f'[{_now()}] logging to {log_path}')
            _process_role(role)
        finally:
            sys.stdout = original_stdout


In [4]:
process_role(ROLE)

[20:07:34] logging to /home/jag/payment-processor-research/logs/new_processing_experian_train.log
[20:07:35] ##### experian/train #####
[20:07:35]   reading split='train' from /home/jag/payment-processor-research/payment_processing_research_data/experian/train/mapped
[20:07:35]   400,000 sample ZEST_KEYs | 443 mapped chunks
[20:07:35]   writing -> /home/jag/payment-processor-research/payment_processing_research_data/new_normalized_and_processed/experian_train/normalized and /home/jag/payment-processor-research/payment_processing_research_data/new_normalized_and_processed/experian_train/processed (dirs cleared)

[20:07:35] phase 1: filter to sample keys + PreprocessorV2 (new aggregator: missing_data_chars=['-'], placeholder=None)
[20:08:56]   chunk 25/443 | kept 404,369/2,500,000 tradelines (16.2%) | 56,605/400,000 keys found | 0.3 chunks/s, ETA 0:22:36
[20:10:15]   chunk 50/443 | kept 798,878/5,000,000 tradelines (16.0%) | 111,842/400,000 keys found | 0.3 chunks/s, ETA 0:20:59
[20:11:3

In [5]:
# sanity: keyed output, joinable back to the sample
proc = pd.read_parquet(OUT_ROOT / f'{BUREAU}_{ROLE}' / 'processed')
old  = pd.read_parquet(DATA_DIR / 'samples' / f'{BUREAU}_{ROLE}' / 'processed_new')
print(f'new_processed {proc.shape}, ZEST_KEY present: '
      f'{proc.index.name == "ZEST_KEY" or "ZEST_KEY" in proc.columns}')
print(f'experiment processed_new (pre-placeholder-change): {old.shape}')

new_processed (394241, 8848), ZEST_KEY present: True
experiment processed_new (pre-placeholder-change): (394241, 8848)
